# Exercise - Multiple Linear Regression



Now apply what you have learned to a new dataset. To keep the context familiar, you will stay with cars and work with the **car seats data**. A company that makes car seats wants a model that predicts its sales, and you will build it.

The file `carseats.csv` is in the `data` folder. It contains 400 observations of the following 11 variables:

* `Sales`: Unit sales (in thousands) at each location
* `CompPrice`: Price charged by the competitor at each location
* `Income`: Community income level (in thousands of dollars)
* `Advertising`: Local advertising budget of the company at each location (in thousands of dollars)
* `Population`: Population size in the region (in thousands)
* `Price`: Price the company charges for car seats at each site
* `ShelveLoc`: A categorical variable (factor) with the levels `Bad`, `Good` and `Medium`, indicating the quality of the shelving location for the car seats at each site
* `Age`: Average age of the local population
* `Education`: Education level at each location
* `Urban`: A categorical variable with the levels `No` and `Yes`, indicating whether the store is in an urban or rural location
* `US`: A categorical variable with the levels `No` and `Yes`, indicating whether the store is in the US or not

> [!NOTE]
> `ShelveLoc`, `Urban` and `US` are text columns, and a regression model needs numbers. Leave these three columns out for now. Notebook 05 (Categorical Variables) shows how to encode them, and you can come back then to test whether they improve your model.

## Learning Objectives

At the end of this notebook, you should be able to:

- Plot a dataset to identify candidate predictors.
- Compare simple linear regression models to find the single best predictor.
- Train a multiple linear regression model on several numeric features.
- Compare models with $R^2$ and adjusted $R^2$.

**(a) Load the data**

In [ ]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


# Set figure stile and size for entire notebook
sns.set_style("ticks")
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
df = pd.read_csv("../data/carseats.csv")
df.head()

**(b) Visualise the data with the appropriate plots.** 

<br>
<details><summary>Show hint</summary>
Check the seaborn documentation for [`pairplot`](https://seaborn.pydata.org/generated/seaborn.pairplot.html). 
</details>

In [ ]:
sns.pairplot(df, corner=True);

**Key trends observed in the pairplot**:
The only clear trend observed is the one between Sales and Price, showing a negative correlation. Other variables show weak or no clear linear relationships with Sales.

**(d) Find the single best predictor for a simple linear regression.**

<br>
<details><summary>Show hint</summary>
Fit one simple linear regression for each numeric feature, compare their $R^2$, and pick the feature with the highest score.
</details>

Documentation: [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).

In [ ]:
X_1 = df[["Price"]]
y_1 = df["Sales"]

In [ ]:
# Create the model
model1 = LinearRegression()

# Fit the  model
model1.fit(X_1, y_1)

# Predict the model
predictions = model1.predict(X_1)

# Print the stats
R_1 = model1.score(X_1, y_1)
print(f"R^2: {R_1}")

**(e) Fit a model with all the numeric explanatory variables.**

In [ ]:
X_2 = df.drop(columns=["Sales", "US", "ShelveLoc", "Urban"])
y_2 = df["Sales"]


# Create the model
model1 = LinearRegression()

# Fit the model
model1.fit(X_2, y_2)

# Predict
predictions = model1.predict(X_2)

# Print the stats
R_2 = model1.score(X_2, y_2)
print(f"R^2: {R_2}")

**(f) What is the best model according to $R^2$?**

In [ ]:
X_2.columns

**(g) Remove a couple of explanatory variables. How does $R^2$ change?**

In [ ]:
X_3 = df.drop(columns=["Sales", "US", "ShelveLoc", "Urban", "Population"])
y_3 = df["Sales"]


# Create the model
model3 = LinearRegression()

# Fit the model
model3.fit(X_3, y_3)

# Predict
predictions = model3.predict(X_3)

R_3 = model3.score(X_3, y_3)
print(f"R^2: {R_3}")

**(h) Repeat steps (f) and (g), this time using the adjusted $R^2$.**

The helper from notebook 03 is defined for you in the next cell.

In [ ]:
def adjusted_r_squared(r_squared, X):
    adjusted_r2 = 1 - ((1 - r_squared) * (len(X) - 1) / (len(X) - X.shape[1] - 1))
    return adjusted_r2

In [ ]:
adj_R_1 = adjusted_r_squared(R_1, X_1)
adj_R_2 = adjusted_r_squared(R_2, X_2)
adj_R_3 = adjusted_r_squared(R_3, X_3)

print(adj_R_1, adj_R_2, adj_R_3)

**(i) What are your most interesting findings?**

In [ ]:
print("=" * 70)
print("MOST INTERESTING FINDINGS FROM THE ANALYSIS")
print("=" * 70)

print("\n1. MODEL PERFORMANCE COMPARISON:")
print(
    f"   - Single Variable (Price only):        R² = {R_1:.4f}, Adj R² = {adj_R_1:.4f}"
)
print(
    f"   - Full Model (7 variables):            R² = {R_2:.4f}, Adj R² = {adj_R_2:.4f}"
)
print(
    f"   - Reduced Model (6 variables):         R² = {R_3:.4f}, Adj R² = {adj_R_3:.4f}"
)

print("\n2. KEY INSIGHTS:")
print("   a) Price alone explains ~20% of sales variation (R² = 0.198)")
print("   b) Adding more variables significantly improves the model:")
print("      - Full model explains 54% of variation (R² = 0.542)")
print("      - This is a 2.7x improvement over the single variable model")

print("\n3. PRACTICAL RECOMMENDATIONS:")
print("   - Best model: 6 variables (CompPrice, Income, Advertising,")
print("     Price, Age, Education)")
print("   - Avoid: Population (adds complexity without value)")
print("   - Price remains the strongest single predictor")
print("   - Always compare adjusted R² when evaluating models with")
print("     different numbers of variables")
print("=" * 70)

## Follow-up: Adding the Categorical Variables

The models above dropped the categorical columns (`ShelveLoc`, `Urban`, `US`). As a follow-up to notebook 05, we bring them back with `pd.get_dummies`, the simplest way to turn categories into 0/1 columns a linear model can use. We pass `drop_first=True` to avoid the dummy variable trap.

Documentation: [`get_dummies`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html).

In [ ]:
# Turn the three categorical columns into 0/1 dummy columns
dummies = pd.get_dummies(df[["ShelveLoc", "Urban", "US"]], drop_first=True)

# Add the dummies to the numeric predictors from the full model (X_2)
X_cat = pd.concat([X_2, dummies], axis=1)
y_cat = df["Sales"]

model_cat = LinearRegression()
model_cat.fit(X_cat, y_cat)

R_cat = model_cat.score(X_cat, y_cat)
adj_R_cat = adjusted_r_squared(R_cat, X_cat)
print(f"R^2 with categoricals:          {R_cat:.4f}")
print(f"Adjusted R^2 with categoricals: {adj_R_cat:.4f}")

Adding the categorical variables raises the R^2 from about 0.54 to about 0.87, a big jump. Almost all of that gain comes from `ShelveLoc`: a Good shelving location is linked to much higher sales. This is the lesson from notebook 05, categorical variables can carry a lot of signal, and `pd.get_dummies` makes them easy to feed into a linear model.

## Summary

In this notebook you:

- Plotted the car seats data to identify candidate predictors of sales.
- Compared simple linear regression models to find the single best predictor.
- Trained multiple linear regression models on several numeric features.
- Compared the models with $R^2$ and adjusted $R^2$, and saw that $R^2$ never falls when you add a feature, while adjusted $R^2$ can.